In [12]:
import kaleidocell
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from datetime import datetime

os.environ['TZ'] = 'Europe/Berlin'
import time
time.tzset()

In [13]:
def checkpoint(msg):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}", flush=True)

In [14]:
def checkpoint(msg):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}", flush=True)

checkpoint("Loading data")
adata = sc.read_h5ad("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/01_extGBmap/immune_cells_extended_gbmap_hgnc.h5ad")
checkpoint(f"Data loaded: {adata.n_obs} cells")

checkpoint("Loading results_mp")
results_mp = kaleidocell.load("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/04_NMF_GSEAonHGNC/04_nmf_all_gsea/results_mp.kc")

checkpoint("Loading mp_scores")
mp_scores = pd.read_csv("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/04_NMF_GSEAonHGNC/04_nmf_all_gsea/mp_scores.csv", index_col=0)
checkpoint(f"mp_scores loaded: {mp_scores.shape[1]} MPs")

[2026-07-06 17:25:36] Loading data
[2026-07-06 17:26:23] Data loaded: 633710 cells
[2026-07-06 17:26:23] Loading results_mp
[2026-07-06 17:26:24] Loading mp_scores
[2026-07-06 17:26:27] mp_scores loaded: 30 MPs


In [15]:
print(mp_scores.columns.tolist())

['MP1_score', 'MP2_score', 'MP3_score', 'MP4_score', 'MP5_score', 'MP6_score', 'MP7_score', 'MP8_score', 'MP9_score', 'MP10_score', 'MP11_score', 'MP12_score', 'MP13_score', 'MP14_score', 'MP15_score', 'MP16_score', 'MP17_score', 'MP18_score', 'MP19_score', 'MP20_score', 'MP21_score', 'MP22_score', 'MP23_score', 'MP24_score', 'MP25_score', 'MP26_score', 'MP27_score', 'MP28_score', 'MP29_score', 'MP30_score']


In [16]:
# ── Select MPs  ───────────────────────────────────────
checkpoint("Selecting top 10 MPs by combined rank")
selected_mps = ['MP4', 'MP2', 'MP11', 'MP10', 'MP3', 'MP12', 'MP20', 'MP5', 'MP30', 'MP1']
checkpoint(f"Selected {len(selected_mps)} MPs: {selected_mps}")

selected_mp_cols = [f"{mp}_score" for mp in selected_mps]
scores_subset = mp_scores[selected_mp_cols]


from scipy.stats import zscore
scores_subset = scores_subset.apply(zscore)

checkpoint(f"Subset shape: {scores_subset.shape}")

[2026-07-06 17:26:27] Selecting top 10 MPs by combined rank
[2026-07-06 17:26:27] Selected 10 MPs: ['MP4', 'MP2', 'MP11', 'MP10', 'MP3', 'MP12', 'MP20', 'MP5', 'MP30', 'MP1']
[2026-07-06 17:26:27] Subset shape: (633710, 10)


In [17]:
# ── Inspect score distributions ─────────────────────────────────────────────
checkpoint("Inspecting score distributions")
print(scores_subset.describe())

scores_subset.hist(bins=50, figsize=(15, 10))
plt.savefig("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/05_significant_MP/05_all_immune_cells/05_2_mp_score_distributions.png", dpi=150, bbox_inches="tight")
plt.close()
checkpoint("Score distribution plot saved")

[2026-07-06 17:26:27] Inspecting score distributions
          MP4_score      MP2_score    MP11_score    MP10_score      MP3_score  \
count  6.337100e+05  633710.000000  6.337100e+05  6.337100e+05  633710.000000   
mean   6.888915e-17       0.000000 -7.750030e-17  2.296305e-17       0.000000   
std    1.000001e+00       1.000001  1.000001e+00  1.000001e+00       1.000001   
min   -1.582521e+00      -1.087598 -1.809281e+00 -1.174849e+00      -1.334442   
25%   -7.723297e-01      -0.545553 -4.712879e-01 -6.348374e-01      -0.780027   
50%   -2.992350e-01      -0.388098 -1.828271e-01 -3.408900e-01      -0.327956   
75%    5.596119e-01      -0.096437  1.783872e-01  2.012741e-01       0.528062   
max    5.340550e+00       4.847317  1.228225e+01  7.186283e+00       4.877458   

         MP12_score    MP20_score     MP5_score    MP30_score     MP1_score  
count  6.337100e+05  6.337100e+05  6.337100e+05  6.337100e+05  6.337100e+05  
mean   1.148153e-17  1.148153e-17  3.444458e-17 -2.583343e-17

In [18]:
# ── Assign dominant MP per cell ─────────────────────────────────────────────
checkpoint("Assigning dominant MP per cell")
adata.obs["dominant_MP"] = scores_subset.idxmax(axis=1).values
max_scores = scores_subset.max(axis=1).values

threshold = np.percentile(max_scores, 75)
checkpoint(f"Using adaptive threshold (75th percentile): {threshold:.4f}")

adata.obs["dominant_MP"] = np.where(max_scores > threshold, adata.obs["dominant_MP"], "Other")
adata.obs["dominant_MP"] = adata.obs["dominant_MP"].astype("category")

checkpoint(f"Distribution:\n{adata.obs['dominant_MP'].value_counts()}")

[2026-07-06 17:26:28] Assigning dominant MP per cell
[2026-07-06 17:26:28] Using adaptive threshold (75th percentile): 2.3779
[2026-07-06 17:26:28] Distribution:
dominant_MP
Other         475282
MP2_score      35577
MP1_score      18313
MP10_score     18065
MP3_score      17516
MP11_score     16637
MP4_score      14743
MP30_score     14511
MP5_score      13061
MP20_score      9959
MP12_score        46
Name: count, dtype: int64


In [19]:
checkpoint(f"obsm keys: {list(adata.obsm.keys())}")
checkpoint(f"obs columns with 'umap' or 'UMAP': {[c for c in adata.obs.columns if 'umap' in c.lower()]}")

if "X_umap" not in adata.obsm:
    checkpoint("X_umap not found — check obs/obsm before recomputing!")
    # Hier NICHT sofort neu rechnen, erst die obigen Logs prüfen

[2026-07-06 17:26:28] obsm keys: ['X_umap']
[2026-07-06 17:26:28] obs columns with 'umap' or 'UMAP': []


In [20]:
#if "X_umap" not in adata.obsm:
    #checkpoint("Computing UMAP")
    #sc.pp.highly_variable_genes(adata, n_top_genes=2000)
    #sc.pp.scale(adata)
    #sc.tl.pca(adata)
    #sc.pp.neighbors(adata)
    #sc.tl.umap(adata)
    #checkpoint("UMAP done")

#checkpoint("Plotting single UMAP colored by dominant MP")
#sc.pl.umap(adata, color="dominant_MP", show=False, palette="tab20")
#plt.savefig("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/05_significant_MP/05_all_immune_cells/05_2_umap_dominant_MP.png", dpi=150, bbox_inches="tight")
#plt.close()
#checkpoint("Saved")

#checkpoint("ALL DONE!")

In [23]:
#adata = sc.read_h5ad("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/01_extGBmap/immune_cells_extended_gbmap_hgnc.h5ad")

print(adata.obs["annotation_level_3"].nunique())
print(adata.obs["annotation_level_3"].value_counts())

10
annotation_level_3
TAM-BDM       267733
TAM-MG        200606
CD4/CD8       101945
Mono           41018
DC              9124
Neutrophil      5117
NK              3151
B cell          2657
Plasma B        1543
Mast             816
Name: count, dtype: int64


In [24]:
checkpoint("Plotting UMAP colored by annotation_level_3")
sc.pl.umap(adata, color="annotation_level_3", show=False, palette="tab20")
plt.savefig("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/05_significant_MP/05_all_immune_cells/05_3_umap_annotation_level_3.png", dpi=150, bbox_inches="tight")
plt.close()
checkpoint("Saved")

[2026-07-06 17:27:21] Plotting UMAP colored by annotation_level_3


[2026-07-06 17:27:25] Saved


In [25]:
checkpoint("Plotting side-by-side comparison")
sc.pl.umap(
    adata,
    color=["dominant_MP", "annotation_level_3"],
    show=False,
    ncols=2,
    wspace=0.4,
    palette="tab20"
)
plt.savefig("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/05_significant_MP/05_all_immune_cells/05_4_umap_MP_vs_annotation.png", dpi=150, bbox_inches="tight")
plt.close()
checkpoint("Saved comparison plot")

[2026-07-06 17:27:25] Plotting side-by-side comparison
[2026-07-06 17:27:32] Saved comparison plot


In [ ]:
crosstab = pd.crosstab(adata.obs["dominant_MP"], adata.obs["annotation_level_3"], normalize="index")
print(crosstab.round(2))

crosstab.to_csv("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/05_significant_MP/05_all_immune_cells/05_6_10mp_vs_annotation_crosstab.csv")

annotation_level_3  B cell  CD4/CD8    DC  Mast  Mono    NK  Neutrophil  \
dominant_MP                                                               
MP10_score            0.00     0.00  0.01   0.0  0.59  0.00        0.01   
MP11_score            0.00     0.22  0.01   0.0  0.00  0.02        0.00   
MP12_score            0.02     0.17  0.02   0.0  0.02  0.02        0.00   
MP1_score             0.00     0.99  0.00   0.0  0.00  0.00        0.00   
MP20_score            0.00     0.00  0.01   0.0  0.41  0.00        0.45   
MP2_score             0.00     0.97  0.00   0.0  0.00  0.03        0.00   
MP30_score            0.00     0.00  0.07   0.0  0.14  0.00        0.00   
MP3_score             0.00     0.15  0.01   0.0  0.02  0.00        0.00   
MP4_score             0.00     0.00  0.00   0.0  0.09  0.00        0.00   
MP5_score             0.00     0.00  0.00   0.0  0.00  0.00        0.00   
Other                 0.01     0.09  0.02   0.0  0.05  0.00        0.00   

annotation_level_3  Plas

In [ ]:
checkpoint("Preparing dominant_MP color mapping (Other = grey)")

categories_ordered = ["Other"] + [c for c in adata.obs["dominant_MP"].cat.categories if c != "Other"]
adata.obs["dominant_MP"] = adata.obs["dominant_MP"].cat.reorder_categories(categories_ordered)

tab20_colors = plt.get_cmap("tab20").colors
color_map = {"Other": "lightgrey"}
mp_idx = 0
for cat in categories_ordered:
    if cat != "Other":
        color_map[cat] = tab20_colors[mp_idx % 20]
        mp_idx += 1

mp_palette = [color_map[cat] for cat in categories_ordered]

checkpoint("Plotting side-by-side comparison (dominant_MP vs annotation_level_3)")
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc.pl.umap(adata, color="dominant_MP", show=False, palette=mp_palette, ax=axes[0], title="Dominant MP")
sc.pl.umap(adata, color="annotation_level_3", show=False, palette="tab20", ax=axes[1], title="Annotation Level 3")

for ax in axes:
    ax.set_aspect("auto")

plt.tight_layout()
plt.savefig("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/05_significant_MP/05_all_immune_cells/05_6_umap_MP_vs_annotation.png", dpi=150, bbox_inches="tight")
plt.close()
checkpoint("Saved comparison plot")

[2026-07-06 17:50:37] Preparing dominant_MP color mapping (Other = grey)
[2026-07-06 17:50:37] Plotting side-by-side comparison (dominant_MP vs annotation_level_3)
[2026-07-06 17:50:45] Saved comparison plot
